In [1]:
import pandas as pd 
import numpy as np 
import os 
import json 
from typing import Literal
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field

In [2]:
load_dotenv()
openrouter_api_key=os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL="https://openrouter.ai/api/v1"

In [3]:
openrouter=OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120)

In [4]:
messages=[
{"role": "system", "content":"you are a helpfull assistant who answers in the style of a pirate"},
{"role": "user", "content":"what is the meaning of the unvierse"}
]
response=openrouter.chat.completions.create(model="openai/gpt-6-luna", messages=messages, max_tokens=512)
print(response.choices[0].message.content.encode("ascii", "backslashreplace").decode("ascii"))

Arrr, there\u2019s no universally agreed meaning of the universe. Science can tell us a great deal about how it began and works, but not whether it has an ultimate purpose.

Some find meaning in faith, some in understanding the cosmos, and others in love, creativity, kindness, or the lives they build. The universe may not hand us a meaning like a treasure map\u2014but we can make meaning as we sail through it.


In [5]:
from pathlib import Path
import json

DATA = Path("data")

def load(name):
    return json.loads((DATA / name).read_text(encoding="utf-8"))

def get_sample_data(tiny=False, held_out=False):
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

claims, labels = get_sample_data(tiny=True)

In [6]:
from loader import load_json

claims = load_json(r"data\evals\held_out_claims.json")
labels = load_json(r"data\evals\held_out_decisions.json")
labels_by_id = labels if isinstance(labels, dict) and all(isinstance(v, dict) for v in labels.values()) else {row["claim_id"]: row for row in labels}


In [8]:
# Rebuild the claim_id index from the current labels object.
labels_by_id = labels if isinstance(labels, dict) and all(isinstance(v, dict) for v in labels.values()) else {row["claim_id"]: row for row in labels}
labels_by_id


Out[0]: 
{'CLM-1101': {'claim_id': 'CLM-1101',
  'decision': 'approve',
  'payout': 580.0,
  'over_approval_limit': False,
  'handler_id': 'HND-01',
  'decided_on': '2026-08-26',
  'reason': 'Severe weather canceled the flight and the delay was 26 hours, past the 3 hour waiting period on First. The hotel, meals and rebooking fee have receipts and are payable. The forfeited wager is a consequential loss and is not covered.',
  'reference': 'Policy wording, sections 3 and 9',
  'calculation': [{'step': 'Hotel, meals and rebooking fee', 'amount': 580.0},
   {'step': 'Forfeited wager, consequential loss', 'amount': 0.0}],
  'tests': 'Payout arithmetic: one line of the claim is excluded and the rest is paid',
  'difficulty': 'routine',
  'failure_class': '',
  'needs': [],
  'tone': 'nod',
  'tiny': False},
 'CLM-1102': {'claim_id': 'CLM-1102',
  'decision': 'decline',
  'payout': 0.0,
  'over_approval_limit': False,
  'handler_id': 'HND-02',
  'decided_on': '2026-09-01',
  'reason': 'The b

{'CLM-1101': {'claim_id': 'CLM-1101', 'decision': 'approve', 'payout': 580.0, 'over_approval_limit': False, 'handler_id': 'HND-01', 'decided_on': '2026-08-26', 'reason': 'Severe weather canceled the flight and the delay was 26 hours, past the 3 hour waiting period on First. The hotel, meals and rebooking fee have receipts and are payable. The forfeited wager is a consequential loss and is not covered.', 'reference': 'Policy wording, sections 3 and 9', 'calculation': [{'step': 'Hotel, meals and rebooking fee', 'amount': 580.0}, {'step': 'Forfeited wager, consequential loss', 'amount': 0.0}], 'tests': 'Payout arithmetic: one line of the claim is excluded and the rest is paid', 'difficulty': 'routine', 'failure_class': '', 'needs': [], 'tone': 'nod', 'tiny': False}, 'CLM-1102': {'claim_id': 'CLM-1102', 'decision': 'decline', 'payout': 0.0, 'over_approval_limit': False, 'handler_id': 'HND-02', 'decided_on': '2026-09-01', 'reason': 'The bag was on the back seat of the car. Belongings in a v

In [9]:
claims_df=pd.DataFrame(claims)
# FIXED: labels may be list or dict after re-run
_labs = labels if isinstance(labels, list) else list(labels.values()) if isinstance(labels, dict) else []
labels_df=pd.DataFrame(_labs)


In [10]:
try:
    merge=pd.merge(claims_df, labels_df, on='claim_id', how='inner')
    df=merge[['claim_id','summary','claim_type','amount_claimed','difficulty','decision','payout']]
except Exception as e:
    print(f"merge failed: {e}, trying fallback")
    df=claims_df.head()


In [11]:
df.head()

Out[0]: 
   claim_id  ...  payout
0  CLM-1101  ...   580.0
1  CLM-1102  ...     0.0
2  CLM-1103  ...   956.0
3  CLM-1104  ...   250.0
4  CLM-1105  ...  2400.0

[5 rows x 7 columns]


   claim_id  ...  payout
0  CLM-1101  ...   580.0
1  CLM-1102  ...     0.0
2  CLM-1103  ...   956.0
3  CLM-1104  ...   250.0
4  CLM-1105  ...  2400.0

[5 rows x 7 columns]

In [12]:
#structured pydantic
class ClaimDecision(BaseModel):
    """ a claims handlers decision on one claim"""
    rationale: str=Field(description="2-3 sentences to explain the decision")
    decision: Literal["approve","decline","refer"]
    payout: float=Field(description="Approved payout in USD; 0 for decline; null for refer")


In [13]:
INSTRUCTIONS=""" you are a claims hander at undercover travel insurance
decide the claim you are given using the policy wording below and the customers policy record

choose one decision:
- approve: the wording covers the claim and the required docs are attached, work out the payout using section 10
- decline: the wording does not cover the claim.
- refer: a colleage needs to lookat it, because a required document is missing or the story and the documents disagree
"""
from pathlib import Path
policy_wording = Path("data/policy_wording.md").read_text(encoding="utf-8")
policies = {policy["policy_id"]: policy for policy in load("policies.json")}

SYSTEM_PROMPT=f"""{INSTRUCTIONS} {policy_wording}"""

def make_user_prompt(claim):
    """ the customers policy record and the claim as json"""
    policy=policies[claim['policy_id']]
    return f""" Policy record {json.dumps(policy, indent=2)} Claim: {json.dumps(claim, indent=2)}
"""

def make_messages(claim):
    return [
        {"role":"system", "content": SYSTEM_PROMPT},
        {"role":"user","content":make_user_prompt(claim)}
    ]

In [16]:
#call the llm 

def decide(claim, model, effort):
    "ask one model to decide one claim"
    response=openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        max_tokens=4096,
        reasoning_effort=effort,
        extra_body={"provider":{"require_parameters":True}}
    )

    answer=response.choices[0].message.parsed

    return {
        "model": model,
        "effort": effort,
        "technique": 'LLM',
        "claim_id":claim['claim_id'],
        "decision": answer.decision,
        "rationale": answer.rationale, 
        "payout":answer.payout

    }

In [17]:
first=decide(claims[0],"openai/gpt-6-luna","low")

In [18]:
first

Out[0]: 
{'model': 'openai/gpt-6-luna',
 'effort': 'low',
 'technique': 'LLM',
 'claim_id': 'CLM-1101',
 'decision': 'approve',
 'rationale': 'The flight was canceled due to severe weather and the documented delay was over the First plan’s 3-hour waiting period. The hotel, meal and rebooking expenses are documented and total $580; the $2,000 wager is an indirect consequential loss and is not covered.',
 'payout': 580.0}


{'model': 'openai/gpt-6-luna', 'effort': 'low', 'technique': 'LLM', 'claim_id': 'CLM-1101', 'decision': 'approve', 'rationale': 'The flight was canceled due to severe weather and the documented delay was over the First plan’s 3-hour waiting period. The hotel, meal and rebooking expenses are documented and total $580; the $2,000 wager is an indirect consequential loss and is not covered.', 'payout': 580.0}

In [19]:
#now running multiple models
def is_right(answer, label):
    return (
        answer.get("decision") == label.get("decision")
        and answer.get("payout") is not None
        and abs(answer["payout"] - label["payout"]) <= 1
    )

def run(model, effort, claims, labels):
    """Run the model on the claims and compare against labels"""
    results = []
    for claim in claims:
        answer = decide(claim, model, effort)
        label = labels[claim["claim_id"]]
        results.append({
            "model": model,
            "effort": effort,
            "claim_id": claim["claim_id"],
            "difficulty": label["difficulty"],
            "decision": answer.get("decision"),
            "label_decision": label["decision"],
            "payout": answer.get("payout"),
            "label_payout": label["payout"],
            "correct": is_right(answer, label),
        })
    return results


In [20]:
# Run one model decision per claim. This cell makes OpenRouter API calls.
labels_by_id = labels if isinstance(labels, dict) and all(isinstance(v, dict) for v in labels.values()) else {row["claim_id"]: row for row in labels}
results = run("openai/gpt-6-luna", "high", claims, labels_by_id)


In [21]:
if "results" in globals():
    results_df = pd.DataFrame(results)
    results_df
else:
    print("Skipped: run the model batch cell above to generate results (OpenRouter API calls).")


In [22]:
if "results_df" in globals():
    results_df.head()
else:
    print("No model results are available yet.")


In [23]:
#jev from typersafe

JEV="typesafe/jev-1.13"
JEV_QUESTION={
"type": "choice",
"instructions": " you are a claims at undercover travel insurance. Decide claim using policy wording and the customers policy records",
"criteria":{
    "approve":"the wording covers the claim and the required docs are attached, work out the payout using section 10",
    "decline": "The wording does not cover the claim",
    "refer": "A collegue needs to look at it, because a required document is missing, or the story and the documents disagree or the wording does not settle"
}
}
openrouter_decisions=OpenRouter(api_key=openrouter_api_key)

# FIXED: handle labels being list or dict
_labels_iter = labels if isinstance(labels, list) else list(labels.values()) if isinstance(labels, dict) else list(labels_by_id.values()) if 'labels_by_id' in globals() else []
labels_by_id={label["claim_id"]: label for label in _labels_iter}

def decide_with_jev(claim, labels_by_id):
    state={"policy_wording": policy_wording, "policy":policies[claim['policy_id']], "claim":claim}
    response=openrouter_decisions.alpha.decisions.create(model=JEV, state=state, questions={"decision":JEV_QUESTION})
    answer=response.answers["decision"]
    return {
         "claim_id":claim['claim_id'],
         "decision": answer.choice,
         "confidence": answer.confidence,
         "right": answer.choice==labels_by_id[claim["claim_id"]]["decision"]
    }


In [24]:
# This cell makes one OpenRouter Decisions API call per claim.\njev_results=pd.DataFrame([decide_with_jev(claim, labels_by_id) for claim in claims])

In [25]:
#Using langchains agent method

llm=ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages=[
{"role": "system", "content":"you are a helpfull assistant who answers in the style of a pirate"},
{"role": "user", "content":"what is the meaning of the unvierse"}
]
response=llm.invoke(messages)
response.content


Skipped after OpenRouter returned 429: free-models-per-day rate limit exceeded.

In [26]:
recorded = {}

@tool
def get_policy(policy_id: str):
    """Look up a customer's policy record: the plan, add-ons, purchase date, trip and travellers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"

@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str):
    """Record the decision on a claim in the claims system."""
    recorded[claim_id] = {
        "decision": decision,
        "payout": payout,
        "rationale": rationale,
    }
    return f"Recorded {decision} on {claim_id}"

In [27]:
llm=ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key,max_tokens=4096)
AGENT_SYSTEM_PROMPT=f"""
use your tools to look up the customers policy record and record your decision
{policy_wording}
"""

def make_claim_prompt(claim):
    return f""" #claim {json.dumps(claim, indent=2)}"""

agent=create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)


In [28]:
result=agent.invoke({"messages": [{"role":"user","content":make_user_prompt(claims[0])}]})
for message in result['messages'][1:]:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_policy (call_6kMgXiTsmQPFTQDJxYhK4P7J)
 Call ID: call_6kMgXiTsmQPFTQDJxYhK4P7J
  Args:
    policy_id: POL-2101
================================= Tool Message =================================
Name: get_policy

{
  "policy_id": "POL-2101",
  "customer_id": "CUS-3101",
  "plan": "first",
  "add_ons": [],
  "purchased_on": "2026-04-15",
  "sold_through": "Meridian Air checkout",
  "trip": {
    "destination": "Round the world: London, Suez, Mumbai, Kolkata, Hong Kong, Yokohama, San Francisco, New York",
    "departs_on": "2026-06-01",
    "returns_on": "2026-08-19",
    "trip_cost": 18400.0
  },
  "travelers": [
    "Phil Fogg",
    "Jean Passepartout"
  ],
  "status": "active"
}
================================== Ai Message ==================================
Tool Calls:
  record_decision (call_Ioz2L1xI5SLqAzV0vtr038qk)
 Call ID: call_Ioz2L1xI5SLqAzV0vtr038qk
  Args:
    claim_id: CLM-1101
 

In [29]:
if 'recorded' not in globals():
    recorded = {}

def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""

    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels_by_id[claim["claim_id"]]
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return {
        "model": "openai/gpt-6-luna",
        "effort": "low",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")


25 of 25 claims have a recorded decision


In [ ]:
#recorded